In [ ]:
# Autoreload para refletir mudanças no config sem reiniciar kernel
%load_ext autoreload
%autoreload 2

In [ ]:
# Célula 2 — stdlib
# terceiros
import joblib
import numpy as np
import pandas as pd
from imblearn.combine import SMOTEENN
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

In [ ]:
# Célula 3 — módulo interno
from churn_telecom.config import (
    DATA_INTERIM,
    DATA_PROCESSED,
    MODELS_DIR,
    RANDOM_STATE,
    TARGET,
    get_logger,
    to_snake_case,
)

In [ ]:
# Célula 4 — configurações de sessão
logger = get_logger("1.03_vab_preprocessing")
logger.info("Iniciando encoding e preprocessing | 1.03_vab_preprocessing")

In [ ]:
# Célula 5 — carrega dataset com features (saída do 1.02)
df = pd.read_parquet(DATA_INTERIM / "telco_features.parquet")
logger.info("Dataset carregado | shape=%s", df.shape)
df.head(5)

In [ ]:
# Célula 6 — validação de pré-condições do 1.02
# ─────────────────────────────────────────────────────────────────────────────
# Verifica três contratos obrigatórios da saída do 1.02:
# 1. Features de engenharia criadas no 1.02 devem existir
# 2. total_charges_log NÃO deve existir — transformação é feita aqui no 1.03
# 3. total_charges original deve existir — é a entrada para o log1p do pipeline
# ─────────────────────────────────────────────────────────────────────────────

# contrato 1 — features do 1.02 presentes
FEATURES_1_02 = [
    "num_services",
    "charges_per_month",
    "is_month_to_month",
    "tenure_group",
    "has_security_support",
    "is_fiber_optic",
]
for feat in FEATURES_1_02:
    assert feat in df.columns, f"Feature '{feat}' ausente — verificar saída do 1.02."

# contrato 2 — total_charges_log NÃO deve existir (transformação é do 1.03)
assert "total_charges_log" not in df.columns, (
    "Coluna 'total_charges_log' encontrada no dataset. "
    "A transformação log1p é responsabilidade do 1.03 — remover do 1.02."
)

# contrato 3 — total_charges original deve existir
assert "total_charges" in df.columns, (
    "Coluna 'total_charges' ausente — necessária para o log1p do pipeline."
)

# contrato 4 — strings originais preservadas (encoding NÃO foi feito no 1.02)
for col in ["online_security", "tech_support", "partner"]:
    assert df[col].dtype in ["object", "category"], (
        f"Coluna '{col}' deve ser string/category mas é {df[col].dtype}. "
        "Encoding é responsabilidade exclusiva do 1.03."
    )

logger.info(
    "Pré-condições validadas | "
    "features_1.02=OK | total_charges_log=ausente (correto) | "
    "total_charges=presente | strings originais preservadas."
)

In [ ]:
# Célula 7 — definição do target e separação X / y
# ─────────────────────────────────────────────────────────────────────────────
# total_charges é mantida — o pipeline aplica log1p internamente.
# Não há colunas a descartar nesta etapa: todas as colunas ou entram
# em algum transformer ou são descartadas pelo remainder="drop".
# ─────────────────────────────────────────────────────────────────────────────
TARGET_COL = to_snake_case(TARGET)

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

logger.info(
    "X shape=%s | y shape=%s | churn_rate=%.2f%%",
    X.shape,
    y.shape,
    y.mean() * 100,
)

In [ ]:
# Célula 8 — definição dos grupos de transformação
# ─────────────────────────────────────────────────────────────────────────────
# ÚNICO ponto de encoding e transformação do projeto.
# Serializado como models/preprocessor.pkl e reutilizado na API FastAPI.
# ─────────────────────────────────────────────────────────────────────────────

# ── Numéricas → log1p + SimpleImputer + StandardScaler ───────────────────────
# log1p aplicado apenas em total_charges (skewness = 0.962)
# Para as demais colunas numéricas, log1p(x) ≈ x quando x é pequeno —
# não distorce, mas aplica uniformemente para simplicidade do pipeline.
# StandardScaler: essencial para MLP (gradientes instáveis sem normalização)
COLS_NUM = [
    "tenure_months",
    "monthly_charges",
    "total_charges",  # log1p aplicado internamente pelo pipeline
    "num_services",
    "charges_per_month",
]

# ── Binárias Yes/No → OrdinalEncoder ─────────────────────────────────────────
# categorias fixas ["No", "Yes"] garantem No=0, Yes=1 independente da ordem
# handle_unknown="use_encoded_value" + unknown_value=-1 — seguro em produção
COLS_BINARY = [
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "paperless_billing",
    "senior_citizen",  # ← movido de COLS_PASSTHROUGH para cá
]


# ── Nominais 3+ categorias → OneHotEncoder ────────────────────────────────────
# drop="first" evita dummy trap (multicolinearidade perfeita)
# handle_unknown="ignore" não quebra se a API receber categoria desconhecida
COLS_OHE = [
    "internet_service",  # DSL / Fiber optic / No
    "contract",  # Month-to-month / One year / Two year
    "payment_method",  # 4 métodos de pagamento
    "gender",  # Male / Female — manter até feature importance
    "tenure_group",  # novo / medio / longo — criada no 1.02
]

# ── Features binárias criadas no 1.02 → passthrough ──────────────────────────
# já são int (0/1) — nenhuma transformação necessária
COLS_PASSTHROUGH = [
    "is_month_to_month",
    "has_security_support",
    "is_fiber_optic",
    # senior_citizen removido daqui
]

logger.info(
    "Grupos | num=%d | binary=%d | ohe=%d | passthrough=%d",
    len(COLS_NUM),
    len(COLS_BINARY),
    len(COLS_OHE),
    len(COLS_PASSTHROUGH),
)

In [ ]:
# Célula 9 — validação de cobertura do ColumnTransformer
# Garante que todas as colunas de X estão em algum grupo — nenhuma perdida
all_grouped = set(COLS_NUM + COLS_BINARY + COLS_OHE + COLS_PASSTHROUGH)
x_cols = set(X.columns)
sem_grupo = x_cols - all_grouped
grupo_inexiste = all_grouped - x_cols

if sem_grupo:
    logger.warning(
        "Colunas em X sem grupo definido (serão descartadas): %s",
        sorted(sem_grupo),
    )
if grupo_inexiste:
    logger.warning(
        "Colunas definidas em grupos mas ausentes em X: %s",
        sorted(grupo_inexiste),
    )
if not sem_grupo and not grupo_inexiste:
    logger.info(
        "Cobertura validada — todas as %d colunas de X cobertas.",
        len(x_cols),
    )

In [ ]:
# Célula 10 — correção no FunctionTransformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    (
                        "log1p",
                        FunctionTransformer(
                            np.log1p,
                            validate=True,
                            feature_names_out="one-to-one",  # ← correção
                        ),
                    ),
                    ("scaler", StandardScaler()),
                ]
            ),
            COLS_NUM,
        ),
        (
            "bin",
            OrdinalEncoder(
                categories=[["No", "Yes"]]
                * len(COLS_BINARY),  # len já inclui senior_citizen
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            COLS_BINARY,
        ),
        (
            "ohe",
            OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore",
            ),
            COLS_OHE,
        ),
        (
            "pass",
            "passthrough",
            COLS_PASSTHROUGH,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [ ]:
df.head(5)

In [ ]:
# Célula 11 — train/test split estratificado
# stratify=y mantém a proporção de churn (~26%) em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

logger.info(
    "Split estratificado | train=%d (%.1f%%) | test=%d (%.1f%%)",
    len(X_train),
    len(X_train) / len(X) * 100,
    len(X_test),
    len(X_test) / len(X) * 100,
)
logger.info(
    "Churn rate | train=%.2f%% | test=%.2f%% | delta=%.3f%%",
    y_train.mean() * 100,
    y_test.mean() * 100,
    abs(y_train.mean() - y_test.mean()) * 100,
)

In [ ]:
# Célula 12 — fit APENAS no treino + transform em ambos
# NUNCA fazer fit no test — vazamento de informação (data leakage)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# força float64 — ColumnTransformer com passthrough retorna dtype object
# quando há mistura de tipos entre transformers
X_train_transformed = X_train_transformed.astype(np.float64)
X_test_transformed = X_test_transformed.astype(np.float64)

feature_names = preprocessor.get_feature_names_out()

logger.info(
    "Transformação concluída | X_train=%s | X_test=%s | n_features=%d | dtype=%s",
    X_train_transformed.shape,
    X_test_transformed.shape,
    len(feature_names),
    X_train_transformed.dtype,
)
logger.info("Features geradas: %s", list(feature_names))

In [ ]:
# Célula diagnóstico — rode antes da Célula 13
import numpy as np

# inspeciona o array cru retornado pelo ColumnTransformer
output_array = X_train_transformed

logger.info("dtype do array transformado: %s", output_array.dtype)
logger.info("shape: %s", output_array.shape)

# encontra quais colunas ainda têm strings
temp_df = pd.DataFrame(output_array, columns=feature_names)
for col in temp_df.columns:
    sample_vals = temp_df[col].dropna().unique()[:3]
    if temp_df[col].dtype == object:
        logger.warning("Coluna object: %s | valores: %s", col, sample_vals)

In [ ]:
# Célula 13 — converte para DataFrame com nomes de features
X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
).astype(float)  # força float — elimina qualquer resíduo de object dtype

X_test_df = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
).astype(float)

# reseta índice — SMOTEENN e sklearn esperam índice sequencial 0..n
X_train_df = X_train_df.reset_index(drop=True)
X_test_df = X_test_df.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# validações pós-transformação
assert X_train_df.isnull().sum().sum() == 0, "Nulos em X_train após transformação"
assert X_test_df.isnull().sum().sum() == 0, "Nulos em X_test após transformação"
assert X_train_df.shape[1] == X_test_df.shape[1], (
    "Número de features diferente entre train e test"
)
assert X_train_df.select_dtypes(include="object").empty, (
    f"Colunas object em X_train: {list(X_train_df.select_dtypes('object').columns)}"
)

logger.info(
    "DataFrames criados | X_train=%s | X_test=%s | dtypes OK",
    X_train_df.shape,
    X_test_df.shape,
)

In [ ]:
# Célula 14 — SMOTE+ENN exclusivamente no treino
# ─────────────────────────────────────────────────────────────────────────────
# SMOTE sozinho causa overfitting leve (Nature Scientific Reports, 2025).
# SMOTE+ENN: oversampling da minoritária + limpeza de fronteira de decisão.
# Aplicado APÓS fit_transform — opera no espaço transformado e normalizado.
# NUNCA aplicar no X_test — contaminaria a avaliação real do modelo.
# ─────────────────────────────────────────────────────────────────────────────
smote_enn = SMOTEENN(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote_enn.fit_resample(X_train_df, y_train)

logger.info(
    "SMOTE+ENN | "
    "antes: n=%d churn=%.2f%% | "
    "depois: n=%d churn=%.2f%% | "
    "delta=%+d amostras",
    len(y_train),
    y_train.mean() * 100,
    len(y_train_res),
    y_train_res.mean() * 100,
    len(y_train_res) - len(y_train),
)

In [ ]:
# Célula 15 — persistência dos datasets processados
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# treino resampled — entrada para baselines e MLP
train_out = X_train_res.copy()
train_out[TARGET_COL] = y_train_res.values
train_out.to_parquet(DATA_PROCESSED / "train.parquet", index=False)

# teste original — NUNCA resampled
test_out = X_test_df.copy()
test_out[TARGET_COL] = y_test.values
test_out.to_parquet(DATA_PROCESSED / "test.parquet", index=False)

# validação pós-save
train_check = pd.read_parquet(DATA_PROCESSED / "train.parquet")
test_check = pd.read_parquet(DATA_PROCESSED / "test.parquet")

assert train_check.shape == train_out.shape, "Erro na persistência do train"
assert test_check.shape == test_out.shape, "Erro na persistência do test"

logger.info(
    "train.parquet | shape=%s | churn=%.2f%%",
    train_out.shape,
    train_out[TARGET_COL].mean() * 100,
)
logger.info(
    "test.parquet  | shape=%s | churn=%.2f%%",
    test_out.shape,
    test_out[TARGET_COL].mean() * 100,
)

In [ ]:
# Célula 16 — serialização do preprocessor
# ─────────────────────────────────────────────────────────────────────────────
# preprocessor.pkl é o mesmo objeto carregado pela API FastAPI na Etapa 3.
# Garante zero divergência entre treino e inferência em produção.
# ─────────────────────────────────────────────────────────────────────────────
MODELS_DIR.mkdir(parents=True, exist_ok=True)
preprocessor_path = MODELS_DIR / "preprocessor.pkl"

joblib.dump(preprocessor, preprocessor_path)

# validação de round-trip — confirma reprodutibilidade da serialização
preprocessor_reloaded = joblib.load(preprocessor_path)
sample = X_test.iloc[:5]
out_original = preprocessor.transform(sample)
out_reloaded = preprocessor_reloaded.transform(sample)

assert np.allclose(out_original, out_reloaded), (
    "Preprocessor recarregado produz resultados diferentes — serialização corrompida."
)

logger.info(
    "Preprocessor serializado e validado | path=%s",
    preprocessor_path,
)

In [ ]:
# Célula 17 — resumo final
logger.info(
    "RESUMO 1.03 | "
    "features_finais=%d | "
    "train_resampled=%d | test_original=%d | "
    "churn_train=%.2f%% | churn_test=%.2f%%",
    X_train_df.shape[1],
    len(y_train_res),
    len(y_test),
    y_train_res.mean() * 100,
    y_test.mean() * 100,
)
logger.info("1.03 preprocessing concluído com sucesso.")